In [7]:
import pandas as pd
import json
import numpy as np
import requests
from tqdm.auto import tqdm
from config import URL, TOKEN
from datetime import datetime
import urllib3
from dateutil.relativedelta import relativedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [9]:
headers = {
    "Authorization": f"Bearer {TOKEN}"
}

start_date = datetime(2017, 1, 1)
today = datetime.now()

all_data = {"entries": []}

current = start_date

while current < today:

    next_month = current + relativedelta(months=1)

    # Do not go beyond today
    month_end = min(next_month, today)

    date_from = current.strftime("%Y%m%d000000")
    date_to = month_end.strftime("%Y%m%d%H%M%S")

    print(f"\nFetching {date_from} -> {date_to}")

    itemCount = np.inf
    offset = 0
    steps = 10
    progress = None

    while offset < itemCount:

        url = (
            f"{URL}assignment/query?"
            f"assignmentType:in:24h;spring;first,"
            f"date:gte:{date_from},"
            f"date:lt:{date_to}"
            f"&sort=startDate.timeUtc"
            f"&limit={steps}"
            f"&skip={offset}"
        )

        response = requests.get(
            url,
            headers=headers,
            verify=False
        )

        if response.status_code != 200:
            print(
                f"Error for {date_from} -> {date_to}:",
                response.status_code
            )
            print(response.text)
            break

        responseData = response.json()
        entries = responseData.get("entries", [])

        if progress is None:
            itemCount = responseData["pagingInfo"]["itemCount"]

            progress = tqdm(
                total=itemCount,
                desc=current.strftime("%Y-%m"),
                unit="entries"
            )

        all_data["entries"].extend(entries)

        progress.update(len(entries))

        # Important: avoid infinite loop
        if len(entries) == 0:
            break

        offset += len(entries)

    if progress is not None:
        progress.close()

    current = next_month


Fetching 20170101000000 -> 20170201000000
Error for 20170101000000 -> 20170201000000: 429
{"title":"BadRequest","status":429,"detail":"Failed to call GetAll at contract.v1.AssignmentProtoService with RequestMsg for ResponseMsg because One or more errors occurred. (Status(StatusCode=\"ResourceExhausted\", Detail=\"Sending message exceeds the maximum configured message size.\"))","instance":"/v1/assignment/query","traceId":"0HNO4TMV3KOFF:00000001"}

Fetching 20170201000000 -> 20170301000000


KeyboardInterrupt: 

In [53]:
exceptions = [
	"metadata",
	"locality",
	"postalcode",
	"contactoptions",
	"personFullName",
	"personFullNameNoTitle",
	"tenantid",
	"persontype",
	"adresse",
	"mailAddressing",
	"socialInsuranceNumber",
	"comments",
	"givenName",
	"familyName",
	"primaryEmailAddress",
	'addresses',
	'primaryPhoneNumber',
	'address',
	'residentialAddress',
	'address',
	'geoLocation',
	'billingAddress',
	'serviceAddress',
	'legalAddress',
	'businessAddress',
	'emailAddresses',
	'phoneNumbers',
	'personFullNameNoTitle',
	'personFullName',
	'mailAddressing',
	'postalAddress',
	'contractualAddressing',
	'caatsDateOfBirth',
	'gender',
	'locations',
	'bankDetails',
	'personStatus',
	"chapterId",
	"language",
	"ordinal",
	"sectionId",
	"academicTitlePrefix",
	"personNr",
	"content",
	"consecutiveNumber",
	'Klient:innen_Empfohlen_Ja',
	'Klient:innen_Erstrkontakt_erfassen_Ja',
	'Klient:innen_Bew_Ein_Ja',
	'abrech-akonto',
	'abrech-buerge-hinterlegt',
	'pers-visite-keine',
	'medizinische-delegation-hochgeladen',
	'medizinische-delegation-nicht-notwendig',
	'Klient:innen_Medizinische_Delegation_Ja',
	'pflegerische-delegation-hochgeladen',
	'pflegerische-delegation-nicht-notwendig',
	'Klient:innen_Pflegerische_Delegation_JA',
	"admin-beruf",
	'pflegevisite-letzte-date',
	'pflegevisite-letzte-dgkp',
	'pflegerisch-person',
	'erstkontakt-bemerkung',
	"admin-kooperationspartner",
	'birthName',
'confessionId',
'confession',
'nationalityId',
'placeOfBirth',
'paymentBlockReason',
'chamberOfCommerceMembershipNr',
'avatarFileId',
'empfohlen-von-category',
'visibilityConditionId',
'deutschkennt-bew-von',
'Kinder_Nein',
'covid-1',
'covid-2',
'covid-3',
'keine-kurse',
'agentur-zuletzt',
'orgRef',
'businesscase',
'comment',
'commentArrival',
'commentDeparture',
'region',
'contactInfo',
'busnessCaseStatus',
'transportOrgArrival',
'transportOrgDeparture',
'assigneeContactInfo',
'assigneeNationality',
'assigneOrgRef',
'assignmentType',
'businessCaseOrgRef',
'businessCaseType',
'assigneeOrgRef',
'hasOrder',
'businessCaseStatus',
"businessCaseStartDate",
"businessCaseEndDate",
"clientAddress",
"region",
"arrivalDate",
"departureDate",
"timeUtc",
"objectKey",
"categoryShortName",
"hasDisplayText",
"displayText",
"objectId",
"objectType",
"cycleLength"
]

exceptions = [x.lower() for x in exceptions]

In [65]:
def extractStatement(statement):
    field = {}
    if statement["statementId"].lower() in exceptions:
        return field

    match statement["answerScheme"]:
        case 1:
            field[statement["statementId"]] = statement["answerValue"]["displayText"]
        case 2:
            #do nothing
            field = {}
        case 3:
            field[statement["statementId"]] = statement["answerYesno"] == 2
        case 4:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False

        case 5:
            answers = statement["answers"]
            for answer in answers:
                if "isSelected" in answer.keys():
                    field[answer["answerId"]] = answer["isSelected"]
                else:
                    field[answer["answerId"]] = False
        case 8:
            if "answerDateTime" in statement.keys():
                if "userLocalTime" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["userLocalTime"]
                elif "timeUtc" in statement["answerDateTime"].keys():
                    field[statement["statementId"]] = statement["answerDateTime"]["timeUtc"]
        case _:
            print(f'Found statement: {statement["answerScheme"]} - {statement["statementId"]} -  {statement}')


    return field
        

def returnEntry(entry):
    keys = entry.keys()
    finalFields = {}
    if "statementId" in keys:
        finalFields = finalFields | extractStatement(entry)
    else:
        for key in keys:
            
            if key.lower() in exceptions:
                continue
            
            typing = type(entry[key]).__name__
            match typing:
                case "str":
                    finalFields[key] = entry[key]
                    
                case "int":
                    finalFields[key] = entry[key]

                case "bool":
                    finalFields[key] = entry[key]

                case "float":
                    finalFields[key] = entry[key]
                    
                case "dict":
                    if key == "assignee":
                        finalFields[key] = entry[key]["objectId"]
                        continue
                    if "displayText" in entry[key].keys():
                        finalFields[key] = entry[key]["displayText"]
                        continue
                    if "userLocalTime" in entry[key].keys():
                        finalFields[key] = entry[key]["userLocalTime"]
                        continue
                    if "timeUtc" in entry[key].keys():
                        finalFields[key] = entry[key]["timeUtc"]
                        continue
                    else:
                        finalFields = finalFields | returnEntry(entry[key])
                    
                case "list":
                    if key == "clients":
                        finalFields[key] = entry[key][0]["objectId"]
                    else:
                        for item in entry[key]:
                            if type(item).__name__ == "dict":
                                finalFields = finalFields | returnEntry(item)
                        

                case _:
                    print(typing)


    
    return finalFields

In [66]:
with open('inputA.json',encoding="utf-8-sig") as json_data:
    data = json.load(json_data)
    json_data.close()

totalData = []

if data["entries"] is not None:
    for entry in data["entries"]:
        test = returnEntry(entry)
        totalData.append(test)
        
df = pd.DataFrame(totalData)
df

,id,startDate,endDate,assignee,billed,clients,admin-packet,arrivalSelfOrganized,departureSelfOrganized,arrivalNoBilling,departureNoBilling,isFirstAssignment
0,401fe723-b4b1-d690-0a0c-cf83552a298e,20220801000000,20261231000000,f3563fe9-0341-c8c3-b361-667663d0a8d3,20260831000000,6b216c49-ce56-f6e9-e064-724ed2fa75f3,Basis,NaN,NaN,NaN,NaN,NaN
1,6dd605e4-bbde-4ebe-c1c4-4e6ac8a48640,20221101000000,20261231000000,7bfbb311-1493-796e-e9d7-72f9d225a6f2,20260831000000,d6e8eb14-ac52-6f5b-88c9-ba41f534da80,CD Perle,True,True,True,True,True
2,019d1ed3-e208-5296-b6d8-8be8db2a2cdf,20241121000000,20261029000000,3699b502-6bb6-b553-764b-b96149fa3ba3,20260831000000,471da45f-2a5e-9b7b-722a-4269e5177271,Premium,NaN,NaN,NaN,NaN,NaN
3,13e90ffc-eb7d-d112-ce97-264e0046f02c,20250904000000,20261001000000,e7929281-32c7-2e82-1865-5acc4ba39f76,20260831000000,f4ca37f2-4cc2-22ba-318f-84555c68ce0d,Demenzbetreuung,NaN,NaN,NaN,NaN,NaN
4,3e3da6cd-ab27-e975-1c2e-697fa82ac698,20250916000000,20260804000000,065f7234-0e82-98a3-2c75-882c8e1479cb,20260803000000,e379d052-96f9-4b8c-ad0d-cf7bde3b54df,Basis,True,True,True,True,NaN
5,d346e8d6-fed2-3ac6-b09d-85c9874c24e1,20251016000000,20270112000000,504c322b-bee1-2460-716a-7af68ffd06c1,20260831000000,45d340df-25e9-d880-dfa2-23214770a711,Basis,True,True,True,True,True
6,fcbefed3-39fc-d41f-93e7-198552a46ec0,20251030000000,20261001000000,61b905a1-25b2-e0df-9a2a-7456d55a962b,20260831000000,b819ba42-1fed-f801-8b80-243900e241fd,Klassik,True,True,True,True,NaN
7,91032c15-d0b2-507f-124e-ef5dd6ade3b5,20251209000000,20260714000000,a8f7fbf3-e50c-6bb4-3811-195fe6fbb4f6,20260713000000,408aa10f-bfae-b4c6-ec0e-f30a90cf9cec,Premium,NaN,NaN,NaN,NaN,NaN
8,bacaca39-0c17-2a0a-7591-0351b473fc9d,20251219000000,20260902000000,6678f396-00f5-385c-7f00-53c48bf8c42a,20260831000000,159a3b11-6126-f597-81ca-ba77b6194bcb,Premium,NaN,NaN,NaN,NaN,True
9,8336d79c-a4b3-0d44-fb6b-fd1d68d3322e,20260107000000,20260728000000,b8e29ed1-70e9-74f7-636a-b08c1d5bbde1,20260727000000,bf912cf2-a5dc-80d8-d039-a32e41b62f5e,Klassik,True,True,True,True,NaN


In [67]:
for col in df.columns:
    non_null = df[col].dropna()

    if len(non_null) > 0 and non_null.isin([True, False]).all():
        df[col] = df[col].fillna(False).astype(bool)

bool_cols = df.select_dtypes(include=["bool", "boolean"]).columns

df[bool_cols] = df[bool_cols].fillna(False)


df = df.replace(r'^\s*$', np.nan, regex=True)

df

,id,startDate,endDate,assignee,billed,clients,admin-packet,arrivalSelfOrganized,departureSelfOrganized,arrivalNoBilling,departureNoBilling,isFirstAssignment
0,401fe723-b4b1-d690-0a0c-cf83552a298e,20220801000000,20261231000000,f3563fe9-0341-c8c3-b361-667663d0a8d3,20260831000000,6b216c49-ce56-f6e9-e064-724ed2fa75f3,Basis,False,False,False,False,False
1,6dd605e4-bbde-4ebe-c1c4-4e6ac8a48640,20221101000000,20261231000000,7bfbb311-1493-796e-e9d7-72f9d225a6f2,20260831000000,d6e8eb14-ac52-6f5b-88c9-ba41f534da80,CD Perle,True,True,True,True,True
2,019d1ed3-e208-5296-b6d8-8be8db2a2cdf,20241121000000,20261029000000,3699b502-6bb6-b553-764b-b96149fa3ba3,20260831000000,471da45f-2a5e-9b7b-722a-4269e5177271,Premium,False,False,False,False,False
3,13e90ffc-eb7d-d112-ce97-264e0046f02c,20250904000000,20261001000000,e7929281-32c7-2e82-1865-5acc4ba39f76,20260831000000,f4ca37f2-4cc2-22ba-318f-84555c68ce0d,Demenzbetreuung,False,False,False,False,False
4,3e3da6cd-ab27-e975-1c2e-697fa82ac698,20250916000000,20260804000000,065f7234-0e82-98a3-2c75-882c8e1479cb,20260803000000,e379d052-96f9-4b8c-ad0d-cf7bde3b54df,Basis,True,True,True,True,False
5,d346e8d6-fed2-3ac6-b09d-85c9874c24e1,20251016000000,20270112000000,504c322b-bee1-2460-716a-7af68ffd06c1,20260831000000,45d340df-25e9-d880-dfa2-23214770a711,Basis,True,True,True,True,True
6,fcbefed3-39fc-d41f-93e7-198552a46ec0,20251030000000,20261001000000,61b905a1-25b2-e0df-9a2a-7456d55a962b,20260831000000,b819ba42-1fed-f801-8b80-243900e241fd,Klassik,True,True,True,True,False
7,91032c15-d0b2-507f-124e-ef5dd6ade3b5,20251209000000,20260714000000,a8f7fbf3-e50c-6bb4-3811-195fe6fbb4f6,20260713000000,408aa10f-bfae-b4c6-ec0e-f30a90cf9cec,Premium,False,False,False,False,False
8,bacaca39-0c17-2a0a-7591-0351b473fc9d,20251219000000,20260902000000,6678f396-00f5-385c-7f00-53c48bf8c42a,20260831000000,159a3b11-6126-f597-81ca-ba77b6194bcb,Premium,False,False,False,False,True
9,8336d79c-a4b3-0d44-fb6b-fd1d68d3322e,20260107000000,20260728000000,b8e29ed1-70e9-74f7-636a-b08c1d5bbde1,20260727000000,bf912cf2-a5dc-80d8-d039-a32e41b62f5e,Klassik,True,True,True,True,False


In [68]:
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [72]:
if "billed" in df.columns and df["billed"].nunique != 2:
	df["billed"] = df["billed"].notna()
if "admin-packet" in df.columns:
	df = pd.get_dummies(df, columns=["admin-packet"], prefix="paket", drop_first=True)


In [73]:
df

,id,startDate,endDate,assignee,billed,clients,arrivalSelfOrganized,departureSelfOrganized,arrivalNoBilling,departureNoBilling,isFirstAssignment,paket_CD Perle,paket_Demenzbetreuung,paket_Klassik,paket_Premium
0,401fe723-b4b1-d690-0a0c-cf83552a298e,20220801000000,20261231000000,f3563fe9-0341-c8c3-b361-667663d0a8d3,True,6b216c49-ce56-f6e9-e064-724ed2fa75f3,False,False,False,False,False,False,False,False,False
1,6dd605e4-bbde-4ebe-c1c4-4e6ac8a48640,20221101000000,20261231000000,7bfbb311-1493-796e-e9d7-72f9d225a6f2,True,d6e8eb14-ac52-6f5b-88c9-ba41f534da80,True,True,True,True,True,True,False,False,False
2,019d1ed3-e208-5296-b6d8-8be8db2a2cdf,20241121000000,20261029000000,3699b502-6bb6-b553-764b-b96149fa3ba3,True,471da45f-2a5e-9b7b-722a-4269e5177271,False,False,False,False,False,False,False,False,True
3,13e90ffc-eb7d-d112-ce97-264e0046f02c,20250904000000,20261001000000,e7929281-32c7-2e82-1865-5acc4ba39f76,True,f4ca37f2-4cc2-22ba-318f-84555c68ce0d,False,False,False,False,False,False,True,False,False
4,3e3da6cd-ab27-e975-1c2e-697fa82ac698,20250916000000,20260804000000,065f7234-0e82-98a3-2c75-882c8e1479cb,True,e379d052-96f9-4b8c-ad0d-cf7bde3b54df,True,True,True,True,False,False,False,False,False
5,d346e8d6-fed2-3ac6-b09d-85c9874c24e1,20251016000000,20270112000000,504c322b-bee1-2460-716a-7af68ffd06c1,True,45d340df-25e9-d880-dfa2-23214770a711,True,True,True,True,True,False,False,False,False
6,fcbefed3-39fc-d41f-93e7-198552a46ec0,20251030000000,20261001000000,61b905a1-25b2-e0df-9a2a-7456d55a962b,True,b819ba42-1fed-f801-8b80-243900e241fd,True,True,True,True,False,False,False,True,False
7,91032c15-d0b2-507f-124e-ef5dd6ade3b5,20251209000000,20260714000000,a8f7fbf3-e50c-6bb4-3811-195fe6fbb4f6,True,408aa10f-bfae-b4c6-ec0e-f30a90cf9cec,False,False,False,False,False,False,False,False,True
8,bacaca39-0c17-2a0a-7591-0351b473fc9d,20251219000000,20260902000000,6678f396-00f5-385c-7f00-53c48bf8c42a,True,159a3b11-6126-f597-81ca-ba77b6194bcb,False,False,False,False,True,False,False,False,True
9,8336d79c-a4b3-0d44-fb6b-fd1d68d3322e,20260107000000,20260728000000,b8e29ed1-70e9-74f7-636a-b08c1d5bbde1,True,bf912cf2-a5dc-80d8-d039-a32e41b62f5e,True,True,True,True,False,False,False,True,False
